# Module 12: Query Performance and Optimization in PostgreSQL

**ALY 6420 | Execution plans, B-tree indexes, and query anti-patterns**

*Course Lecture Notes*

## Module 12
### Moving from correctness to performance
Module 11 introduced text matching and GIN indexes. Module 12 investigates why different access paths have different costs. A correct query can still be too slow for a production workload.

## Learning objectives
- Interpret EXPLAIN and EXPLAIN ANALYZE plans, including estimates, timing and loops.
- Explain B-tree access, selectivity and the planner's cost-based choice.
- Rewrite non-sargable predicates, unnecessary projections and correlated subqueries where appropriate.
- Diagnose slow SQL using evidence, verify equivalent output and communicate trade-offs.

## Principal source and database
Shan et al. (2025), *SQL for Data Analytics*, 4th ed., Chapter 10, **Performant SQL**, pp. 187–207. 
Use PostgreSQL 16 in DBeaver with the Neon-hosted `sqlda` database. All SQL is in Markdown blocks for copying into DBeaver. Run temporary-table examples in a single session, in order. 

## First step: inspect the schema

```sql
SELECT current_database(), current_schema(), version();
SELECT table_schema, table_name, column_name, data_type
FROM information_schema.columns
WHERE table_name IN ('customers','sales','emails')
ORDER BY table_schema, table_name, ordinal_position;
SELECT schemaname, tablename, indexname, indexdef
FROM pg_indexes WHERE tablename IN ('customers','sales','emails')
ORDER BY schemaname, tablename, indexname;
```

Reference dataset: A deterministic synthetic workload is generated to make comparisons possible without access to ZoomZoom. It contains 2,000 customers and 40,000 sales, intentionally skewed states and a wide text field. Customers 1901 - 2000 have no purchases. 

```sql
CREATE TEMP TABLE m12_customers AS
SELECT i AS customer_id,
       'Customer ' || i AS first_name,
       CASE WHEN i % 200 = 0 THEN 'Rivera' ELSE 'Name ' || (i % 100) END AS last_name,
       CASE WHEN i % 100 = 0 THEN 'TX' ELSE 'CA' END AS state,
       'User' || i || '@example.com' AS email,
       repeat(md5(i::text), 20) AS notes
FROM generate_series(1,2000) AS g(i);
CREATE TEMP TABLE m12_sales AS
SELECT i AS transaction_id, 1 + (i-1)%1900 AS customer_id,
       DATE '2020-01-01' + (i%1096) AS transaction_date,
       (50 + i%900)::numeric AS sales_amount
FROM generate_series(1,40000) AS g(i);
ANALYZE m12_customers;
ANALYZE m12_sales;
```

```sql
SELECT state, count(*) FROM m12_customers GROUP BY state ORDER BY state;
SELECT count(*) AS sales, count(DISTINCT customer_id) AS buyers FROM m12_sales;
```

Expected: CA 1,980; TX 20; sales 40,000; buyers 1,900. If rerunning, first drop only the temporary tables created in your own session using the cleanup at the end.

# Part 1: Reading Query Execution Plans

```sql
EXPLAIN SELECT customer_id, last_name FROM m12_customers WHERE state = 'TX';
EXPLAIN (ANALYZE, BUFFERS)
SELECT customer_id, last_name FROM m12_customers WHERE state = 'TX';
```

## Anatomy of a plan
Illustrative output:

```text
Seq Scan on m12_customers (cost=0.00..210.00 rows=20 width=12)
  (actual time=0.020..0.450 rows=20 loops=1)
  Filter: (state = 'TX'::text)
  Rows Removed by Filter: 1980
Planning Time: 0.100 ms
Execution Time: 0.470 ms
```

| Field | Interpretation |
|---|---|
| Node | Operation performed |
| cost | Estimated startup and completion work in relative units, not milliseconds |
| rows | Estimated output rows, not rows examined |
| width | Estimated bytes per output row |
| actual time | Time to first and last output, averaged per execution |
| actual rows / loops | Average output per execution and number of executions |

The example emits 20 rows but examines 2,000. Parent timing includes child work; do not add all node times. A LIMIT may stop a child early. Read the indentation to follow dependencies, not as a literal chronological schedule.

```sql
EXPLAIN SELECT * FROM emails;
EXPLAIN SELECT * FROM emails LIMIT 5;
EXPLAIN SELECT clicked_date FROM emails
WHERE clicked_date >= DATE '2021-01-01' AND clicked_date < DATE '2022-01-01';
```

Dates here are illustrative; inspect the actual date distribution. Plain EXPLAIN does not execute the statement. EXPLAIN ANALYZE executes it, including writes; use read-only SELECT examples in this lecture.

### Supplementary self-paced example 12.1: Inspect estimate and execution

**Goal and context.** Read child actual rows, not just root rows.

**Predict → run → inspect.** Before running, predict the output. Copy the SQL into DBeaver (PostgreSQL 16); run temporary-table examples in one session. If interrupted, remove only the named temporary table before retrying.

```sql
EXPLAIN (ANALYZE, BUFFERS) SELECT count(*) FROM generate_series(1,1000) n WHERE n%10=0;
```

**Expected result:** Aggregate returns one row; 100 input rows qualify.

**Derived practice:** Change divisor to 20. Predict the changed result before running.

<details>
<summary>Reveal self-check</summary>

50 qualify; timing varies.

</details>

**Transfer:** Explain which assumption would need checking before applying this technique to production data. Record the result and compare it with your prediction.

## Scan types and buffer evidence

| Access path | When it can help |
|---|---|
| Seq Scan | Large fraction of rows or small relation |
| Index Scan | Targeted access, often selective conditions |
| Bitmap Index + Heap Scan | Collect row locations and visit heap pages together |
| Index Only Scan | Needed values are in the index; visibility checks may still require heap fetches |

`BUFFERS` distinguishes cached hits and blocks read into PostgreSQL buffers; a read need not be physical disk I/O. Temporary relations report local buffers. Avoid summing inclusive parent/child counters.

## Join strategies

```sql
EXPLAIN (ANALYZE, BUFFERS)
SELECT c.state, count(*) AS purchases
FROM m12_customers c JOIN m12_sales s USING(customer_id)
GROUP BY c.state;
```

- Nested Loop: repeats inner access for outer rows, useful with selective indexed lookups.
- Hash Join: builds a hash table from one input, probes it with the other for equality matches.
- Merge Join: walks ordered join inputs; may require sorts.

Look for `Hash Cond`, `Merge Cond`, `Join Filter` and repeated inner loops. A join may use more than one access method. Estimated/actual row discrepancies can reflect skew, correlated columns, expression selectivity or stale statistics. Refresh statistics first when appropriate; it is not proof of the cause.

```sql
SELECT relname, n_live_tup, last_analyze, last_autoanalyze
FROM pg_stat_user_tables WHERE relname IN ('customers','sales');
SELECT attname, n_distinct, most_common_vals, most_common_freqs
FROM pg_stats WHERE tablename = 'm12_customers' AND attname = 'state';
```

Knowledge check: a SubPlan shows actual time 0.010..0.200, rows=1, loops=2000. Approximate cumulative time is 400 ms, not 0.2 ms. This is inclusive node time, not a separate addition to total execution time.

### Example 12.2: Join grain and access strategy

**Goal and context.** Equality joins permit several strategies; observe actual loops.

**Predict → run → inspect.** Before running, predict the output. Copy the SQL into DBeaver (PostgreSQL 16); run temporary-table examples in one session. If interrupted, remove only the named temporary table before retrying.

```sql
EXPLAIN (ANALYZE, BUFFERS) SELECT a.i,b.i FROM generate_series(1,20) a(i) JOIN generate_series(1,100) b(i) ON a.i=b.i;
```

**Expected result:** 20 joined rows; planner chooses strategy.

**Derived practice:** Change first series to 50. Predict the changed result before running.

<details>
<summary>Reveal self-check</summary>

50 joined rows.

</details>

**Transfer:** Explain which assumption would need checking before applying this technique to production data. Record the result and compare it with your prediction.

# Part 2: How B-tree Indexes Work

A B-tree maintains ordered keys and references to heap tuples. Internal pages route lookups; leaf pages contain keys and row references. All leaves are at the same depth. The tree grows slowly in height, but heap fetches and the number of matches still matter.

```text
                  [separator: M]
                  /            \
          [A ... L]          [M ... Z]
             |                   |
        ordered leaf keys -> heap tuple references
```
Conceptual diagram, not an exact page layout. The heap need not be in index order.

```sql
EXPLAIN (ANALYZE, BUFFERS)
SELECT customer_id, last_name FROM m12_customers WHERE state='TX';
CREATE INDEX m12_state_idx ON m12_customers(state);
EXPLAIN (ANALYZE, BUFFERS)
SELECT customer_id, last_name FROM m12_customers WHERE state='TX';
EXPLAIN (ANALYZE, BUFFERS)
SELECT customer_id, last_name FROM m12_customers WHERE state='CA';
```

TX matches 1%; CA matches 99%. Compare the access paths, buffers and measured times. An index scan or bitmap scan is plausible for TX; a sequential scan is plausible for CA. There is no universal selectivity threshold. Table width, clustering, cache and costs matter.

Compare state, gender and latitude filters, showing why equality alone does not guarantee index efficiency. Check the following read-only checks if the columns exist.

```sql
SELECT state, count(*) FROM customers GROUP BY state ORDER BY count(*);
SELECT gender, count(*) FROM customers GROUP BY gender;
EXPLAIN SELECT customer_id FROM customers WHERE latitude > 30 AND latitude < 38;
```

## Equality, range, prefix and ordering

```sql
CREATE INDEX m12_last_idx ON m12_customers(last_name text_pattern_ops);
EXPLAIN SELECT customer_id FROM m12_customers WHERE last_name LIKE 'River%';
EXPLAIN SELECT customer_id FROM m12_customers WHERE last_name LIKE '%vera';
CREATE INDEX m12_date_idx ON m12_sales(transaction_date);
EXPLAIN SELECT transaction_id, transaction_date FROM m12_sales
ORDER BY transaction_date LIMIT 10;
```

Prefix LIKE and anchored regex can be index-eligible with suitable collation/operator class; a leading wildcard cannot provide an ordinary B-tree prefix range. Pattern operator classes may not support ordinary locale-dependent ordering. GIN with pg_trgm is an alternative for many substring searches (Module 11).

## Composite indexes and leftmost columns

```sql
CREATE INDEX m12_customer_date_idx ON m12_sales(customer_id, transaction_date);
EXPLAIN SELECT transaction_id FROM m12_sales WHERE customer_id=42;
EXPLAIN SELECT transaction_id FROM m12_sales WHERE customer_id=42
AND transaction_date >= DATE '2022-01-01';
EXPLAIN SELECT transaction_id FROM m12_sales WHERE transaction_date >= DATE '2022-01-01';
```

A leading equality plus a range on the next key provides efficient bounds. In PostgreSQL 16 a non-leading condition can sometimes use a multicolumn index, but generally cannot narrow the scanned portion efficiently without the leading key. The standalone date index is another candidate here.

### Example 12.3: Composite index for device readings

**Goal and context.** Leading equality plus range matches the key order.

**Predict → run → inspect.** Before running, predict the output. Copy the SQL into DBeaver (PostgreSQL 16); run temporary-table examples in one session. If interrupted, remove only the named temporary table before retrying.

```sql
CREATE TEMP TABLE m12_readings AS SELECT i AS id,i%100 AS device_id, date '2025-01-01'+(i%365) AS day FROM generate_series(1,50000) i;
CREATE INDEX ON m12_readings(device_id,day);
ANALYZE m12_readings;
EXPLAIN (ANALYZE,BUFFERS) SELECT id FROM m12_readings WHERE device_id=7 AND day>=date '2025-06-01';
DROP TABLE m12_readings;
```

**Expected result:** Plan and timing vary; only device 7 and qualifying dates.

**Derived practice:** Remove device condition. Predict the changed result before running.

<details>
<summary>Reveal self-check</summary>

Many more rows; compare buffers, do not assume index usage.

</details>

**Transfer:** Explain which assumption would need checking before applying this technique to production data. Record the result and compare it with your prediction.

## Partial and expression indexes

```sql
CREATE INDEX m12_tx_partial_idx ON m12_customers(customer_id) WHERE state='TX';
EXPLAIN SELECT customer_id FROM m12_customers WHERE state='TX' AND customer_id > 1000;
CREATE INDEX m12_email_lower_idx ON m12_customers(lower(email));
EXPLAIN SELECT customer_id FROM m12_customers WHERE lower(email)='user42@example.com';
```

A partial index is useful only when the query implies its predicate. An expression index stores the transformed value. The textbook latitude partial-index example (p. 205) illustrates targeting a recurring workload. Indexes cost storage and maintenance; not every UPDATE requires changing every index (HOT updates are possible).

## Hash indexes and other index types

```sql
CREATE INDEX m12_email_hash_idx ON m12_customers USING hash(email);
EXPLAIN SELECT customer_id FROM m12_customers WHERE email='User42@example.com';
EXPLAIN SELECT customer_id FROM m12_customers WHERE email > 'User42@example.com';
```

Exercise 10.3 (pp. 201–204) contrasts hash and B-tree access. Hash supports equality, not range or ordering. Activity 10 compares equality and range on sales.customer_id; repeat that experiment on an isolated copy if desired.

| Type | Typical use |
|---|---|
| B-tree | Ordered scalar comparisons |
| Hash | Equality |
| GIN | Full-text, JSONB and arrays with suitable operators |
| BRIN | Large physically correlated tables |
| GiST / SP-GiST | Specialized search structures, e.g. spatial data |

Production note: `CREATE INDEX CONCURRENTLY` permits ordinary writes during most of the build but still takes locks, needs extra work, and cannot run inside a transaction block. Do not test production DDL casually.

### Example 12.4: Partial and expression index tradeoffs

**Goal and context.** Partial and expression indexes support different predicate shapes.

**Predict → run → inspect.** Before running, predict the output. Copy the SQL into DBeaver (PostgreSQL 16); run temporary-table examples in one session. If interrupted, remove only the named temporary table before retrying.

```sql
CREATE TEMP TABLE m12_jobs AS SELECT i AS id, CASE WHEN i%100=0 THEN 'Queued' ELSE 'Done' END AS status FROM generate_series(1,10000) i;
CREATE INDEX ON m12_jobs(id) WHERE status='Queued';
CREATE INDEX ON m12_jobs(lower(status));
ANALYZE m12_jobs;
EXPLAIN SELECT id FROM m12_jobs WHERE status='Queued';
EXPLAIN SELECT id FROM m12_jobs WHERE lower(status)='queued';
DROP TABLE m12_jobs;
```

**Expected result:** Each predicate matches 100 rows; plans vary.

**Derived practice:** Search lower(status)=done. Predict the changed result before running.

<details>
<summary>Reveal self-check</summary>

9900 rows; index may not be selected.

</details>

**Transfer:** Explain which assumption would need checking before applying this technique to production data. Record the result and compare it with your prediction.

# Part 3: Query Anti-Patterns and Rewrites

## Sargability: search argument able
A predicate is sargable relative to an available index when it can be used as an index search condition. Distinguish eligibility from actual selection.

```sql
EXPLAIN (ANALYZE, BUFFERS)
SELECT transaction_id FROM m12_sales WHERE EXTRACT(YEAR FROM transaction_date)=2022;
EXPLAIN (ANALYZE, BUFFERS)
SELECT transaction_id FROM m12_sales WHERE transaction_date >= DATE '2022-01-01'
AND transaction_date < DATE '2023-01-01';
```

Both queries should return the same IDs. A year covers many rows here, so both might use Seq Scan. Try a one-day range next. For timestamptz, define the intended time zone before rewriting calendar boundaries.

```sql
EXPLAIN (ANALYZE, BUFFERS)
SELECT transaction_id FROM m12_sales WHERE transaction_date + 1 = DATE '2022-06-16';
EXPLAIN (ANALYZE, BUFFERS)
SELECT transaction_id FROM m12_sales WHERE transaction_date = DATE '2022-06-15';
```

## Validate equivalence before claiming improvement

```sql
WITH a AS (SELECT transaction_id FROM m12_sales WHERE EXTRACT(YEAR FROM transaction_date)=2022),
b AS (SELECT transaction_id FROM m12_sales WHERE transaction_date >= DATE '2022-01-01' AND transaction_date < DATE '2023-01-01')
SELECT 'a minus b' AS difference, count(*) FROM (SELECT * FROM a EXCEPT ALL SELECT * FROM b) d
UNION ALL
SELECT 'b minus a', count(*) FROM (SELECT * FROM b EXCEPT ALL SELECT * FROM a) d;
```

Expected: zero differences both ways. EXCEPT ALL checks multiplicity, unlike ordinary EXCEPT. It does not check order; use explicit ORDER BY with tie-breakers when order matters.

### Example 12.5: Validate a timestamp rewrite

**Goal and context.** Half-open ranges preserve fractional-second boundaries.

**Predict → run → inspect.** Before running, predict the output. Copy the SQL into DBeaver (PostgreSQL 16); run temporary-table examples in one session. If interrupted, remove only the named temporary table before retrying.

```sql
WITH t(d) AS (VALUES (timestamp '2025-02-01'),(timestamp '2025-02-28 23:59:59'),(timestamp '2025-03-01')), a AS (SELECT d FROM t WHERE date_trunc('month',d)=timestamp '2025-02-01'), b AS (SELECT d FROM t WHERE d>=timestamp '2025-02-01' AND d<timestamp '2025-03-01') SELECT * FROM ((SELECT * FROM a EXCEPT ALL SELECT * FROM b) UNION ALL (SELECT * FROM b EXCEPT ALL SELECT * FROM a)) diff;
```

**Expected result:** 0 differences.

**Derived practice:** Use <= March 1. Predict the changed result before running.

<details>
<summary>Reveal self-check</summary>

One extra row appears.

</details>

**Transfer:** Explain which assumption would need checking before applying this technique to production data. Record the result and compare it with your prediction.

## SELECT * and unnecessary projection

```sql
EXPLAIN (ANALYZE, BUFFERS) SELECT * FROM m12_customers WHERE state='CA';
EXPLAIN (ANALYZE, BUFFERS) SELECT customer_id, last_name FROM m12_customers WHERE state='CA';
```

Compare estimated width and plan shape. A narrower projection reduces returned data and may enable index-only access; it does not necessarily reduce heap page reads. EXPLAIN ANALYZE discards result rows, so it does not measure network transfer savings.

## Correlated subqueries

```sql
EXPLAIN (ANALYZE, BUFFERS)
SELECT c.customer_id,
 (SELECT SUM(s.sales_amount) FROM m12_sales s WHERE s.customer_id=c.customer_id) AS total
FROM m12_customers c;
EXPLAIN (ANALYZE, BUFFERS)
SELECT c.customer_id, t.total
FROM m12_customers c LEFT JOIN
 (SELECT customer_id, SUM(sales_amount) AS total FROM m12_sales GROUP BY customer_id) t
 USING(customer_id);
```

The rewrite aggregates at customer grain before joining. Expected 2,000 rows, including 100 NULL totals. Preserve LEFT JOIN semantics; SUM for no sales is NULL, whereas a scalar COUNT is zero. For COUNT use COALESCE on a preaggregated count. An indexed correlated lookup may be efficient for a small outer input; measure instead of assuming every join is faster.

```sql
WITH a AS (SELECT c.customer_id, (SELECT SUM(s.sales_amount) FROM m12_sales s WHERE s.customer_id=c.customer_id) AS total FROM m12_customers c),
b AS (SELECT c.customer_id,t.total FROM m12_customers c LEFT JOIN (SELECT customer_id,SUM(sales_amount) AS total FROM m12_sales GROUP BY customer_id) t USING(customer_id))
SELECT count(*) AS differences FROM ((SELECT * FROM a EXCEPT ALL SELECT * FROM b) UNION ALL (SELECT * FROM b EXCEPT ALL SELECT * FROM a)) d;
```

Expected: 0. Compare SubPlan loops, aggregate strategy and execution time.

### Example 12.6: Preaggregate machine downtime

**Goal and context.** Aggregate once per machine, then retain machines without events.

**Predict → run → inspect.** Before running, predict the output. Copy the SQL into DBeaver (PostgreSQL 16); run temporary-table examples in one session. If interrupted, remove only the named temporary table before retrying.

```sql
WITH machines(id) AS (VALUES (1),(2),(3)), events(id,minutes) AS (VALUES (1,10),(1,20),(2,5)), totals AS (SELECT id,sum(minutes) AS mins FROM events GROUP BY id) SELECT m.id,t.mins FROM machines m LEFT JOIN totals t USING(id) ORDER BY m.id;
```

**Expected result:** 1:30, 2:5, 3:NULL.

**Derived practice:** Use INNER JOIN. Predict the changed result before running.

<details>
<summary>Reveal self-check</summary>

Machine 3 disappears.

</details>

**Transfer:** Explain which assumption would need checking before applying this technique to production data. Record the result and compare it with your prediction.

## Implicit and explicit type conversions

```sql
CREATE INDEX m12_id_idx ON m12_customers(customer_id);
EXPLAIN SELECT customer_id FROM m12_customers WHERE customer_id::text='42';
EXPLAIN SELECT customer_id FROM m12_customers WHERE customer_id=42;
EXPLAIN SELECT customer_id FROM m12_customers WHERE customer_id='42';
```

The last literal initially has unknown type and is normally resolved to integer, so it can use the index. It is not equivalent to casting the indexed column to text. A comparison of a text column to a typed integer may be rejected rather than implicitly converted. Match parameter and column types explicitly.

### Example 12.7: Parameter type discipline

**Goal and context.** Bind values with the column type rather than casting indexed columns.

**Predict → run → inspect.** Before running, predict the output. Copy the SQL into DBeaver (PostgreSQL 16); run temporary-table examples in one session. If interrupted, remove only the named temporary table before retrying.

```sql
SELECT pg_typeof(42), pg_typeof('42'::text), pg_typeof('42'::integer);
```

**Expected result:** integer, text, integer.

**Derived practice:** Try casting abc to integer separately. Predict the changed result before running.

<details>
<summary>Reveal self-check</summary>

Invalid input error; validate inputs before query.

</details>

**Transfer:** Explain which assumption would need checking before applying this technique to production data. Record the result and compare it with your prediction.

# Part 4: Technique Selection and Diagnostic Workflow

## Three questions before optimizing
1. **Is there actually a problem?** Measure execution time against the workload requirement. A fast query may need no change.
2. **Are statistics current?** Compare estimates and actual rows, refresh with ANALYZE when justified, then measure again.
3. **Is it the query or the schema?** Inspect access conditions, repeated work, projection and available indexes. Change one factor at a time.

The detailed diagnostic sequence is used here.

```sql
ANALYZE m12_sales;
EXPLAIN (ANALYZE, BUFFERS) SELECT transaction_id FROM m12_sales
WHERE customer_id=42 AND transaction_date >= DATE '2022-01-01';
```

## Record evidence
Repeat comparable tests several times, alternate run order where possible, retain identical parameters and record cache conditions. Do not clear shared caches on a hosted database.

| Metric | Baseline | Revised |
|---|---|---|
| Node / Index Cond | | |
| Estimated / actual rows | | |
| Loops | | |
| Buffer hits / reads | | |
| Execution time (ms) | | |
| Equivalent result? | | |

Speedup = baseline time / revised time. Percentage time reduction = 100 × (baseline − revised) / baseline. Never substitute planner cost for measured execution time.

# Part 5: Applied ZoomZoom Analytics

Business question: Which regions account for sales in a selected period? Begin with a correct aggregate; inspect the join and range access.

```sql
EXPLAIN (ANALYZE, BUFFERS)
SELECT c.state, COUNT(*) AS purchases, SUM(s.sales_amount) AS revenue
FROM customers c JOIN sales s ON c.customer_id=s.customer_id
WHERE s.transaction_date >= DATE '2020-06-01'
AND s.transaction_date < DATE '2020-07-01'
GROUP BY c.state ORDER BY revenue DESC;
```

Verify schema and choose a period with data. Run the SELECT without EXPLAIN to inspect results. A sale is assumed to be one transaction row; check grain before interpreting purchase counts. Examine whether the date filter is selective, whether a join key index exists, and whether the aggregate sorts or hashes. Do not add an index unless the measured workload warrants it.

### Example 12.8: AI query tuning evidence

**Goal and context.** An AI suggestion to index everything is not evidence. Capture baseline, result equality, representative workload and write overhead.

**Predict → run → inspect.** Before running, predict the output. Copy the SQL into DBeaver (PostgreSQL 16); run temporary-table examples in one session. If interrupted, remove only the named temporary table before retrying.

```sql
EXPLAIN (ANALYZE,BUFFERS) SELECT count(*) FROM generate_series(1,10000) n WHERE n>9990;
```

**Expected result:** 10 qualifying rows; aggregate returns 1 row.

**Derived practice:** Change threshold to 100. Predict the changed result before running.

<details>
<summary>Reveal self-check</summary>

9900 qualifying rows; compare plan evidence.

</details>

**Transfer:** Explain which assumption would need checking before applying this technique to production data. Record the result and compare it with your prediction.

# Part 6: Common Errors and AI-Generated SQL Critique

Consider this invented AI response:

> The query has cost 800, so it takes 800 ms. A new index on state guarantees an Index Scan for CA. Replacing the correlated SUM with SELECT DISTINCT removes repeated work.

Critique: cost is not elapsed time; CA matches almost all reference customers, so scanning may be cheaper; DISTINCT does not remove the correlated calculation and can change output grain. Require actual plans and equivalent results rather than trusting confident language.

Other traps: treating estimate error as proof of stale statistics, ignoring loops, comparing different result sets, summing inclusive node times, and assuming an index is free to maintain.

# Part 7: Practice and Knowledge Checks

## Practice Questions
1. Compare unrestricted and LIMIT plans on `m12_sales`. Explain startup vs total cost.
2. Compare TX and CA filters and record selectivity. Explain any bitmap access.
3. Compare the one-day predicate before and after rewriting; verify with EXCEPT ALL.
4. Compare an indexed lookup for customer 42 with a broad range of customers. Explain why plans may differ.
5. Adapt the correlated SUM example to AVG. Preserve customers with no sales and validate both directions.
6. Propose a partial index for a recurring date window; explain maintenance and predicate requirements. Use only temporary data.


## Knowledge check
| Question | reasoning |
|---|---|
| Is cost 500 equal to 500 ms? | No, relative planner units. |
| Why might a B-tree not be chosen? | Low selectivity, small table, cost estimates or unsuitable predicate. |
| Does rows=20 mean only 20 rows were scanned? | No, it is output cardinality. |
| Does a faster INNER JOIN always replace a scalar aggregate? | No, it can remove customers with no matches. |
| Does ANALYZE create indexes? | No, it updates planner statistics. |
| Can a normal B-tree handle arbitrary suffix searches efficiently? | Not as a prefix lookup; consider appropriate alternative indexing. |

# Summary
- Measure first; costs and actual timings are different.
- Read plan trees through their dependencies and interpret loops and cardinality.
- B-tree indexes support ordered lookup; selection is cost-based.
- Rewrite predicates and repeated work only when semantics are preserved.
- Indexes trade read efficiency for storage and maintenance.
- Validate output and document observed performance, not promised speedups.

Next: Module 13, recursive CTEs and statistical SQL.

## References
- Shan, J., Li, H., Goldwasser, M., Malik, U., & Johnston, B. (2025). *SQL for Data Analytics* (4th ed.). Packt. Chapter 10, pp. 187–207.
- ALY 6420: Introduction to Querying. Canvas Module 12, sections 12.0–12.7.
- PostgreSQL Global Development Group. [14.1 Using EXPLAIN](https://www.postgresql.org/docs/16/using-explain.html); [EXPLAIN command](https://www.postgresql.org/docs/16/sql-explain.html); [11.2 Index Types](https://www.postgresql.org/docs/16/indexes-types.html).
- Additional technical references: [Multicolumn Indexes](https://www.postgresql.org/docs/16/indexes-multicolumn.html), [Indexes on Expressions](https://www.postgresql.org/docs/16/indexes-expressional.html), [Partial Indexes](https://www.postgresql.org/docs/16/indexes-partial.html), [CREATE INDEX](https://www.postgresql.org/docs/16/sql-createindex.html).
- Winand, M. [SQL Indexing and Tuning e-book](https://use-the-index-luke.com/)

